# Prepare empirical DEEP features

Run the cells in order to convert prepared VCFs into 50-kb haplotype matrices,
calculate human haplotype and combined human/Denisovan diploid statistics,
and merge the chromosome tables for prediction.

Use Python 3.10–3.12 on Linux. From the repository root, install the VCF dependencies
and a notebook interface with `python -m pip install ".[vcf]" jupyterlab`.
Then open this notebook with JupyterLab and select that Python environment.

## Input panel

Supply one sorted, BGZF-compressed and indexed VCF per GRCh37/hg19 autosome.
This notebook expects a single contig in each VCF header, named `1` through `22`
or `chr1` through `chr22`.
Each file must contain only the target humans and one Denisovan, with biallelic SNPs,
complete diploid calls, and phased human heterozygotes. The Denisovan may be unphased.
The study panels contained seven humans (14 human haplotypes) plus one Denisovan.
Other panel sizes require independent calibration of model scores.

Complete cohort selection, singleton ascertainment in the appropriate larger panel,
and accessibility/site masking before running these cells. The code below begins
with the prepared cohort VCFs. It does not perform those upstream steps.

## Set input and output paths

Replace the `/path/to/...` placeholders or set the corresponding environment
variables before starting Jupyter. Use a separate output directory for each cohort.
`VCF_PATTERN` should match only the prepared files for that cohort. Set `CHROMOSOMES`
to the autosomes to process, for example `[22]` for a single-chromosome run.
Output files with the same names are overwritten on reruns.

In [ ]:
import os
from pathlib import Path

# Input: your prepared, indexed, single-chromosome cohort VCFs.
VCF_DIRECTORY = Path(os.environ.get(
    "DEEP_EMPIRICAL_VCF_DIR", "/path/to/your/prepared_cohort_vcfs"
))
VCF_PATTERN = "*.vcf.gz"
DENISOVA_ID = "Denisova"  # Replace with the Denisovan sample ID in your VCFs.
CHROMOSOMES = list(range(1, 23))

# Output: choose a writable directory for this cohort's matrices and feature tables.
OUTPUT_DIRECTORY = Path(os.environ.get(
    "DEEP_EMPIRICAL_OUTPUT_DIR", "/path/to/your/empirical_deep_output"
))
HAPLOTYPE_DIRECTORY = OUTPUT_DIRECTORY / "haplotype_matrices"
FEATURE_DIRECTORY = OUTPUT_DIRECTORY / "features"

for directory in (VCF_DIRECTORY, OUTPUT_DIRECTORY):
    if "/path/to/" in directory.as_posix():
        raise ValueError("Replace the input and output path placeholders before running.")
if not VCF_DIRECTORY.is_dir():
    raise FileNotFoundError(f"Prepared VCF directory not found: {VCF_DIRECTORY}")

## 1. Build haplotype matrices

Each row is a retained SNP. Column 0 contains its 1-based genomic position,
followed by two adjacent haplotype columns for each human in VCF sample order,
and finally the two Denisovan alleles. Seven humans produce 17 columns in total.
Rows with no alternate allele are omitted at this step. Human-monomorphic sites
are removed when statistics are calculated.

The NPZ archive stores one matrix per non-empty window. Keys such as `1_50000`
are 1-based, inclusive coordinates; the final window is clipped to chromosome length.
GRCh37 chromosome lengths are specified below. A whole chromosome's matrices are
held in memory before the archive is written.

In [ ]:
import re
from cyvcf2 import VCF
import numpy as np
import os

WINDOW_SIZE = 50000

# GRCh37/hg19 autosome lengths, keyed by chromosome number.
CHROM_LENGTHS = {
    "1": 249250621,
    "2": 243199373,
    "3": 198022430,
    "4": 191154276,
    "5": 180915260,
    "6": 171115067,
    "7": 159138663,
    "8": 146364022,
    "9": 141213431,
    "10": 135534747,
    "11": 135006516,
    "12": 133851895,
    "13": 115169878,
    "14": 107349540,
    "15": 102531392,
    "16": 90354753,
    "17": 81195210,
    "18": 78077248,
    "19": 59128983,
    "20": 63025520,
    "21": 48129895,
    "22": 51304566
}

def get_contig_length(vcf_file, chrom, vcf_reader, lengths_dict=None):
    """
    Get the contig length from the provided lengths_dict if available.
    Otherwise, attempt to extract it from the header or by scanning the records.
    """
    chrom_number = chrom.removeprefix('chr')
    if lengths_dict is not None and chrom_number in lengths_dict:
        return lengths_dict[chrom_number]
    
    # Try to extract from the header.
    m = re.search(r'##contig=<ID={0},length=(\d+)>'.format(re.escape(chrom)), vcf_reader.raw_header)
    if m:
        return int(m.group(1))
    else:
        # Fallback: open a temporary VCF reader and scan for the maximum record.end.
        max_end = 0
        temp_vcf = VCF(vcf_file)
        for record in temp_vcf:
            if record.end > max_end:
                max_end = record.end
        temp_vcf.close()
        return max_end

def process_window(records, human_indices, den_idx):
    """
    Build the haplotype matrix for a list of VCF records (one window), including a position column.
    
    For each variant record:
      - The first column is the variant position (record.POS).
      - The next columns: for human samples (phased genotypes, e.g. "0|1"), each genotype is split into two alleles.
      - The last two columns come from the Denisovan's unphased genotype.
      
    For seven human individuals (14 haplotypes) and one Denisovan,
    the resulting matrix has 1 position + 14 human + 2 Denisovan = 17 columns.
    
    Only rows that have at least one variant allele (a "1") in the genotype calls are included.
    """
    rows = []
    for record in records:
        # Build the genotype data portion (without the position)
        genotype_calls = []
        # Process human samples (phased genotypes).
        for hi in human_indices:
            gt = record.genotypes[hi]  # Expected format: [allele1, allele2, phased?]
            genotype_calls.extend([gt[0], gt[1]])
        # Process the Denisovan sample (unphased genotype).
        den_gt = record.genotypes[den_idx]
        genotype_calls.extend([den_gt[0], den_gt[1]])
        
        # Check if there is at least one variant allele (a "1") among the genotype calls.
        # Skip an all-reference row; human polymorphism is checked during feature extraction.
        if not any(allele == 1 for allele in genotype_calls):
            continue
        
        # Include the variant position as the first column.
        row = [record.POS] + genotype_calls
        rows.append(row)
    
    if rows:
        return np.array(rows, dtype=int)
    else:
        # Preserve the matrix layout even when no SNPs remain in this window.
        return np.empty((0, 1 + 2 * len(human_indices) + 2), dtype=int)

def process_vcf_windows(vcf_file, window_size=WINDOW_SIZE, denisova_id="Denisova", out_dir=".", lengths_dict=None):
    """Save the non-empty windows from one indexed, single-contig autosomal VCF."""
    vcf_file = str(vcf_file)
    vcf_reader = VCF(vcf_file)
    try:
        samples = vcf_reader.samples
        if denisova_id not in samples or len(samples) < 2:
            raise ValueError("The VCF must contain the named Denisovan and target humans.")
        den_idx = samples.index(denisova_id)
        human_indices = [i for i in range(len(samples)) if i != den_idx]

        # Region queries use the one autosome declared in this file's header.
        seqnames = vcf_reader.seqnames
        if len(seqnames) != 1 or seqnames[0].removeprefix("chr") not in CHROM_LENGTHS:
            raise ValueError("Use one GRCh37 autosome per VCF, with a single contig in its header.")
        contig = seqnames[0]
        chrom = contig.removeprefix("chr")
        lengths = CHROM_LENGTHS if lengths_dict is None else lengths_dict
        total_length = get_contig_length(vcf_file, contig, vcf_reader, lengths_dict=lengths)
        window_matrices = {}

        for window_start in range(1, total_length + 1, window_size):
            window_end = min(window_start + window_size - 1, total_length)
            region_str = f"{contig}:{window_start}-{window_end}"
            matrix = process_window(list(vcf_reader(region_str)), human_indices, den_idx)
            if matrix.shape[0] == 0:
                continue
            window_matrices[f"{window_start}_{window_end}"] = matrix
    finally:
        vcf_reader.close()

    os.makedirs(out_dir, exist_ok=True)
    npz_filename = Path(out_dir) / f"haplotype_matrices_{chrom}.npz"
    np.savez(npz_filename, **window_matrices)
    print(f"Chromosome {chrom}: saved {len(window_matrices)} non-empty windows to {npz_filename}")
    return npz_filename

In [ ]:
# Inspect the matched files before processing. Each selected chromosome must appear once.
vcf_files = sorted(VCF_DIRECTORY.glob(VCF_PATTERN))
if not vcf_files:
    raise FileNotFoundError(f"No VCFs match {VCF_DIRECTORY / VCF_PATTERN}")
vcf_by_chromosome = {}
for vcf_file in vcf_files:
    reader = VCF(str(vcf_file))
    try:
        contigs = reader.seqnames
    finally:
        reader.close()
    if len(contigs) != 1 or contigs[0].removeprefix("chr") not in CHROM_LENGTHS:
        raise ValueError(f"Expected one GRCh37 autosome in the header of {vcf_file}")
    chrom = int(contigs[0].removeprefix("chr"))
    if chrom not in CHROMOSOMES:
        continue
    if chrom in vcf_by_chromosome:
        raise ValueError(f"More than one VCF matched chromosome {chrom}")
    vcf_by_chromosome[chrom] = vcf_file
missing = set(CHROMOSOMES) - set(vcf_by_chromosome)
if missing:
    raise FileNotFoundError(f"No prepared VCFs for chromosomes {sorted(missing)}; check CHROMOSOMES and VCF_PATTERN.")

matrix_files = {}
for chrom, vcf_file in sorted(vcf_by_chromosome.items()):
    print(f"Processing {vcf_file}")
    matrix_files[chrom] = process_vcf_windows(
        vcf_file, denisova_id=DENISOVA_ID, out_dir=HAPLOTYPE_DIRECTORY
    )

## 2. Calculate window statistics

The functions below fold alleles across humans and the Denisovan, retain sites
that are polymorphic in humans, and calculate haploid human statistics and diploid
statistics for the combined panel. Denisovan-conditioned features summarize sharing,
distances, concentration, and clustering along each window.

The minor-allele-count retention probabilities `p1`, `p2`, and `p3` are all 1,
so this step retains every eligible site. Within-individual statistics use 10-kb
bins, and Denisovan-sharing occupancy uses 5-kb bins. The `top_*` differences are
reported from largest to smallest. Feature names and their ordering matter when
using a trained model.

In [ ]:
import numpy as np
from scipy.stats import skew, kurtosis
from scipy.spatial.distance import pdist, squareform

def tajimas_d_from_counts(k_counts, n_chrom):
    """Compute Tajima's D from allele counts and the number of haplotypes."""
    seg = (k_counts > 0) & (k_counts < n_chrom)
    S = np.sum(seg)
    if n_chrom < 2 or S == 0:
        return 0.0

    a1 = np.sum(1.0 / np.arange(1, n_chrom))
    a2 = np.sum(1.0 / (np.arange(1, n_chrom) ** 2))
    b1 = (n_chrom + 1) / (3 * (n_chrom - 1))
    b2 = 2 * (n_chrom**2 + n_chrom + 3) / (9 * n_chrom * (n_chrom - 1))
    c1 = b1 - 1 / a1
    c2 = b2 - (n_chrom + 2) / (a1 * n_chrom) + a2 / (a1**2)
    e1 = c1 / a1
    e2 = c2 / (a1**2 + a2)

    k = k_counts.astype(float)
    pi_site = 2.0 * k * (n_chrom - k) / (n_chrom * (n_chrom - 1))
    pi_total = np.sum(pi_site)
    theta_w = S / a1
    var_d = e1 * S + e2 * S * (S - 1)

    return float((pi_total - theta_w) / np.sqrt(var_d)) if var_d > 0 else 0.0


def _gini(x):
    """
    Compute the Gini coefficient of a 1D array x.
    Returns 0.0 if x is empty or all zeros.
    """
    x = np.asarray(x, float)
    if x.size == 0:
        return 0.0
    mean_x = x.mean()
    if mean_x == 0:
        return 0.0
    x_sorted = np.sort(x)
    n = x_sorted.size
    g = (2.0 * np.sum((np.arange(1, n + 1) * x_sorted)) / (n * np.sum(x_sorted))) - (n + 1) / n
    return float(g)


def _entropy_from_counts(counts):
    counts = np.asarray(counts, dtype=float)
    total = counts.sum()
    if total <= 0:
        return 0.0
    p = counts / total
    p = p[p > 0]
    return float(-np.sum(p * np.log(p)))


def _safe_moments(x):
    x = np.asarray(x, dtype=float)
    if x.size == 0:
        return 0.0, 0.0, 0.0, 0.0
    return (
        float(np.mean(x)),
        float(np.var(x)),
        float(np.nan_to_num(skew(x), nan=0.0)),
        float(np.nan_to_num(kurtosis(x), nan=0.0)),
    )


def _safe_quantile(x, q):
    x = np.asarray(x, dtype=float)
    if x.size == 0:
        return 0.0
    return float(np.quantile(x, q))


def _longest_run_info(positions, binary_vec):
    """
    binary_vec: 1D array of 0/1 values in site order
    positions: corresponding absolute positions

    Returns:
      longest_run_sites, longest_run_bp, num_runs, mean_run_sites
    """
    positions = np.asarray(positions)
    binary_vec = np.asarray(binary_vec).astype(int)

    if binary_vec.size == 0 or np.sum(binary_vec) == 0:
        return 0, 0.0, 0, 0.0

    one_idx = np.where(binary_vec == 1)[0]
    if one_idx.size == 0:
        return 0, 0.0, 0, 0.0

    runs = []
    start = one_idx[0]
    prev = one_idx[0]

    for idx in one_idx[1:]:
        if idx == prev + 1:
            prev = idx
        else:
            runs.append((start, prev))
            start = idx
            prev = idx
    runs.append((start, prev))

    run_sites = [b - a + 1 for a, b in runs]
    run_bp = []
    for a, b in runs:
        if a == b:
            run_bp.append(1.0)
        else:
            run_bp.append(float(positions[b] - positions[a]))

    return (
        int(np.max(run_sites)),
        float(np.max(run_bp)),
        int(len(runs)),
        float(np.mean(run_sites)),
    )


def _compute_denisovan_conditioned_hap_features(human_positions, human_geno, den_haps, total_length_bp=50000):
    """
    Denisovan-conditioned features on human haplotypes.

    human_positions: absolute genomic coordinates after human-only filtering
    human_geno:      (n_sites, n_human_haps)
    den_haps:        (n_sites, 2) Denisovan haplotypes aligned to the same rows
    """
    stats = {}

    n_sites = human_geno.shape[0]
    n_haps = human_geno.shape[1]

    if n_sites == 0 or den_haps is None or den_haps.shape[0] != n_sites:
        zero_keys = [
            "den_shared_sites",
            "den_shared_sites_h1",
            "den_shared_sites_h2",
            "den_shared_sites_h3_4",
            "den_shared_sites_h5_7",
            "den_shared_sites_h8plus",
            "mean_human_count_given_den_shared",
            "var_human_count_given_den_shared",
            "min_hap_den_distance",
            "mean_hap_den_distance",
            "var_hap_den_distance",
            "q10_hap_den_distance",
            "top1_closest_hap_den",
            "top2_closest_hap_den",
            "top3_closest_hap_den",
            "mean_den_shared_per_hap",
            "var_den_shared_per_hap",
            "skew_den_shared_per_hap",
            "kurt_den_shared_per_hap",
            "max_den_shared_per_hap",
            "gini_den_shared_per_hap",
            "entropy_den_shared_per_hap",
            "den_shared_nn_mean",
            "den_shared_nn_var",
            "den_shared_nn_skew",
            "den_shared_nn_kurt",
            "den_shared_max_sites_in_5kb",
            "den_shared_num_occupied_5kb_windows",
            "den_shared_longest_run_bp_all",
            "den_shared_longest_run_sites_all",
            "mean_longest_run_bp_per_hap",
            "var_longest_run_bp_per_hap",
            "mean_longest_run_sites_per_hap",
            "var_longest_run_sites_per_hap",
            "max_longest_run_bp_per_hap",
            "max_longest_run_sites_per_hap",
            "mean_num_runs_per_hap",
            "var_num_runs_per_hap",
            "mean_run_size_per_hap",
            "var_run_size_per_hap",
        ]
        for k in zero_keys:
            stats[k] = 0.0
        return stats

    den_present = np.any(den_haps == 1, axis=1)
    human_counts = np.sum(human_geno, axis=1)

    # 1) Denisovan-conditioned site-frequency features
    den_shared_mask = den_present & (human_counts > 0)
    den_shared_counts = human_counts[den_shared_mask]

    stats["den_shared_sites"] = int(np.sum(den_shared_mask))
    stats["den_shared_sites_h1"] = int(np.sum(den_shared_counts == 1))
    stats["den_shared_sites_h2"] = int(np.sum(den_shared_counts == 2))
    stats["den_shared_sites_h3_4"] = int(np.sum((den_shared_counts >= 3) & (den_shared_counts <= 4)))
    stats["den_shared_sites_h5_7"] = int(np.sum((den_shared_counts >= 5) & (den_shared_counts <= 7)))
    stats["den_shared_sites_h8plus"] = int(np.sum(den_shared_counts >= 8))

    stats["mean_human_count_given_den_shared"] = float(np.mean(den_shared_counts)) if den_shared_counts.size > 0 else 0.0
    stats["var_human_count_given_den_shared"] = float(np.var(den_shared_counts)) if den_shared_counts.size > 0 else 0.0

    # 2) Human haplotype ↔ Denisovan affinity
    dists = []
    for h in range(n_haps):
        hvec = human_geno[:, h]
        d1 = np.mean(hvec != den_haps[:, 0]) * n_sites
        d2 = np.mean(hvec != den_haps[:, 1]) * n_sites
        dists.append(min(d1, d2))
    dists = np.asarray(dists, dtype=float)

    stats["min_hap_den_distance"] = float(np.min(dists)) if dists.size > 0 else 0.0
    stats["mean_hap_den_distance"] = float(np.mean(dists)) if dists.size > 0 else 0.0
    stats["var_hap_den_distance"] = float(np.var(dists)) if dists.size > 0 else 0.0
    stats["q10_hap_den_distance"] = _safe_quantile(dists, 0.10)

    sorted_d = np.sort(dists)
    stats["top1_closest_hap_den"] = float(sorted_d[0]) if sorted_d.size >= 1 else 0.0
    stats["top2_closest_hap_den"] = float(sorted_d[1]) if sorted_d.size >= 2 else 0.0
    stats["top3_closest_hap_den"] = float(sorted_d[2]) if sorted_d.size >= 3 else 0.0

    # 3) Concentration of Denisovan-shared alleles across human haplotypes
    if np.sum(den_shared_mask) > 0:
        shared_per_hap = np.sum(human_geno[den_shared_mask, :], axis=0).astype(float)
    else:
        shared_per_hap = np.zeros(n_haps, dtype=float)

    m, v, s, k = _safe_moments(shared_per_hap)
    stats["mean_den_shared_per_hap"] = m
    stats["var_den_shared_per_hap"] = v
    stats["skew_den_shared_per_hap"] = s
    stats["kurt_den_shared_per_hap"] = k
    stats["max_den_shared_per_hap"] = float(np.max(shared_per_hap)) if shared_per_hap.size > 0 else 0.0
    stats["gini_den_shared_per_hap"] = _gini(shared_per_hap)
    stats["entropy_den_shared_per_hap"] = _entropy_from_counts(shared_per_hap)

    # 4) Positional clustering of Denisovan-shared sites
    den_shared_pos = np.sort(human_positions[den_shared_mask])

    if den_shared_pos.size >= 2:
        nn = np.diff(den_shared_pos)
        m, v, s, k = _safe_moments(nn)
        stats["den_shared_nn_mean"] = m
        stats["den_shared_nn_var"] = v
        stats["den_shared_nn_skew"] = s
        stats["den_shared_nn_kurt"] = k
    else:
        stats["den_shared_nn_mean"] = 0.0
        stats["den_shared_nn_var"] = 0.0
        stats["den_shared_nn_skew"] = 0.0
        stats["den_shared_nn_kurt"] = 0.0

    if den_shared_pos.size > 0:
        # 5 kb occupancy on absolute coordinates
        bins_5kb = (den_shared_pos // 5000).astype(int)
        uniq_bins, counts_bins = np.unique(bins_5kb, return_counts=True)
        stats["den_shared_max_sites_in_5kb"] = int(np.max(counts_bins))
        stats["den_shared_num_occupied_5kb_windows"] = int(uniq_bins.size)
    else:
        stats["den_shared_max_sites_in_5kb"] = 0
        stats["den_shared_num_occupied_5kb_windows"] = 0

    # 5) Consecutive shared-allele runs in the retained site order
    longest_bp = []
    longest_sites = []
    num_runs = []
    mean_run_sites = []

    if np.sum(den_shared_mask) > 0:
        den_shared_positions = human_positions[den_shared_mask]
        den_shared_human = human_geno[den_shared_mask, :]

        global_longest_bp = 0.0
        global_longest_sites = 0

        for h in range(n_haps):
            l_sites, l_bp, n_runs, mean_sites = _longest_run_info(
                den_shared_positions,
                den_shared_human[:, h]
            )
            longest_bp.append(l_bp)
            longest_sites.append(l_sites)
            num_runs.append(n_runs)
            mean_run_sites.append(mean_sites)

            if l_bp > global_longest_bp:
                global_longest_bp = l_bp
            if l_sites > global_longest_sites:
                global_longest_sites = l_sites

        stats["den_shared_longest_run_bp_all"] = float(global_longest_bp)
        stats["den_shared_longest_run_sites_all"] = int(global_longest_sites)
    else:
        longest_bp = [0.0] * n_haps
        longest_sites = [0.0] * n_haps
        num_runs = [0.0] * n_haps
        mean_run_sites = [0.0] * n_haps
        stats["den_shared_longest_run_bp_all"] = 0.0
        stats["den_shared_longest_run_sites_all"] = 0

    stats["mean_longest_run_bp_per_hap"] = float(np.mean(longest_bp))
    stats["var_longest_run_bp_per_hap"] = float(np.var(longest_bp))
    stats["mean_longest_run_sites_per_hap"] = float(np.mean(longest_sites))
    stats["var_longest_run_sites_per_hap"] = float(np.var(longest_sites))
    stats["max_longest_run_bp_per_hap"] = float(np.max(longest_bp)) if len(longest_bp) > 0 else 0.0
    stats["max_longest_run_sites_per_hap"] = float(np.max(longest_sites)) if len(longest_sites) > 0 else 0.0
    stats["mean_num_runs_per_hap"] = float(np.mean(num_runs))
    stats["var_num_runs_per_hap"] = float(np.var(num_runs))
    stats["mean_run_size_per_hap"] = float(np.mean(mean_run_sites))
    stats["var_run_size_per_hap"] = float(np.var(mean_run_sites))

    return stats


def extract_statistics_both(genotypes, total_length_bp=50000, window_size_bp=10000,
                            top_within_sample=5, top_num_diff=5):
    """
    Run both human-only (phased) and combined (diploid) analyses on the genotype matrix.

    Parameters:
        genotypes (numpy.ndarray): 2D array with rows as variant sites and columns as follows:
            - Column 0 holds variant positions.
            - Columns 1 through -3: human haplotype calls.
            - Last 2 columns: Denisovan genotype calls.
        total_length_bp (int): Nominal length of the window (default 50 kb).
        window_size_bp (int): Bin width (in bp) for the within-sample statistics.
        top_within_sample (int): Number of top within-window differences to report.
        top_num_diff (int): Number of top pairwise differences to report.

    Returns:
        dict with:
          - "human_only_stats"
          - "combined_stats"
    """
    positions = genotypes[:, 0]
    genotype_data = genotypes[:, 1:]

    # Encode the less frequent allele across humans and the Denisovan as 1.
    def reencode_minor_major(geno):
        geno = geno.astype(np.int16)
        reencoded = np.empty_like(geno)

        for i in range(geno.shape[0]):
            site = geno[i]
            derived_count = np.sum(site)
            if derived_count > geno.shape[1] / 2:
                reencoded[i] = np.where(site == 1, 0, 1)
            else:
                reencoded[i] = site

        return reencoded

    genotype_data = reencode_minor_major(genotype_data)

    # All three retention probabilities are 1: no additional site thinning.
    n = genotype_data.shape[1]
    ac = genotype_data.sum(axis=1)
    mac = np.minimum(ac, n - ac)

    rng = np.random.default_rng()
    p1 = 1
    p2 = 1
    p3 = 1

    p = np.ones_like(mac, float)
    p[mac == 1] = p1
    p[mac == 2] = p2
    p[mac >= 3] = p3

    keep = rng.random(len(mac)) < p
    keep = (np.random.random(len(mac)) < p)

    genotype_data = genotype_data[keep]
    positions = positions[keep]

    has_one = np.any(genotype_data == 1, axis=1)
    has_zero = np.any(genotype_data == 0, axis=1)
    mask = has_one & has_zero
    genotype_data = genotype_data[mask]
    positions = positions[mask]

    # Human-only filtered matrix
    raw_human = genotype_data[:, :-2]
    raw_den = genotype_data[:, -2:]

    has_one_hum = np.any(raw_human == 1, axis=1)
    has_zero_hum = np.any(raw_human == 0, axis=1)
    nonzero_mask = has_one_hum & has_zero_hum

    human_positions = positions[nonzero_mask]
    human_genotypes = raw_human[nonzero_mask, :]
    den_at_human_sites = raw_den[nonzero_mask, :]

    human_stats = _compute_haploid_stats(
        human_positions,
        human_genotypes,
        total_length_bp,
        window_size_bp,
        top_within_sample,
        top_num_diff,
        den_haps=den_at_human_sites
    )

    genotype_data = genotype_data[nonzero_mask, :]
    positions = positions[nonzero_mask]

    if genotype_data.shape[0] == 0:
        return None

    # Combined diploid analysis
    num_human_haps = genotype_data.shape[1] - 2
    if num_human_haps % 2 != 0:
        raise ValueError("The number of human haplotypes (excluding Denisovan) must be even.")

    human_diploids = np.sum(
        genotype_data[:, :num_human_haps].reshape(genotype_data.shape[0], -1, 2),
        axis=2
    )
    den_dip = np.sum(genotype_data[:, -2:], axis=1, keepdims=True)
    combined_diploids = np.hstack([human_diploids, den_dip])

    combined_stats = _compute_diploid_stats(
        positions,
        combined_diploids,
        total_length_bp,
        window_size_bp,
        top_within_sample,
        top_num_diff
    )

    return {"human_only_stats": human_stats, "combined_stats": combined_stats}


def _compute_haploid_stats(positions, geno, total_length_bp, window_size_bp,
                           top_within_sample, top_num_diff, den_haps=None):
    """
    Compute summary statistics for haploid (phased) genotype data.

    positions: absolute genomic coordinates
    geno:      (n_variants, n_haplotypes)
    den_haps:  optional aligned Denisovan haplotypes (n_variants, 2)
    """
    stats = {}
    total_sites, num_haplotypes = geno.shape
    stats['num_sites'] = total_sites
    stats['num_haplotypes'] = num_haplotypes

    if total_sites == 0:
        stats['mean_allele_frequency'] = 0.0
        stats['var_allele_frequency'] = 0.0
        stats['heterozygosity'] = 0.0
        stats['prop_segregating_sites'] = 0.0
        stats['site_frequency_spectrum'] = [0] * (num_haplotypes + 1)
        stats['tajimas_d'] = 0.0
        stats['num_singletons'] = 0
        stats['mean_pairwise_diff'] = 0.0
        stats['var_pairwise_diff'] = 0.0
        stats['pairwise_diff_skewness'] = 0.0
        stats['pairwise_diff_kurtosis'] = 0.0
        for i in range(1, top_num_diff + 1):
            stats[f'top_pairwise_diff_{i}'] = 0.0

        if den_haps is not None:
            den_stats = _compute_denisovan_conditioned_hap_features(
                positions, geno, den_haps, total_length_bp=total_length_bp
            )
            for k, v in den_stats.items():
                stats[k] = v
        return stats

    allele_freq = np.mean(geno, axis=1)
    stats['mean_allele_frequency'] = float(np.mean(allele_freq))
    stats['var_allele_frequency'] = float(np.var(allele_freq))

    heterozygosity = 2 * allele_freq * (1 - allele_freq)
    stats['heterozygosity'] = float(np.mean(heterozygosity))

    segregating = np.logical_and(np.any(geno > 0, axis=1), np.any(geno == 0, axis=1))
    S_total = np.sum(segregating)
    stats['prop_segregating_sites'] = float(S_total) / total_sites if total_sites > 0 else 0.0

    site_variances = np.std(geno, axis=1)
    polymorphic_idx = np.where(site_variances > 0)[0]

    derived_counts = np.sum(geno, axis=1)
    sfs = np.bincount(derived_counts.astype(np.int64), minlength=num_haplotypes + 1)
    stats['site_frequency_spectrum'] = sfs.tolist()

    stats['tajimas_d'] = tajimas_d_from_counts(derived_counts, n_chrom=geno.shape[1])
    stats['num_singletons'] = int(sfs[1]) if len(sfs) > 1 else 0

    pdist_values = pdist(geno.T, metric='hamming') * total_sites
    stats['mean_pairwise_diff'] = float(pdist_values.mean()) if pdist_values.size > 0 else 0.0
    stats['var_pairwise_diff'] = float(pdist_values.var()) if pdist_values.size > 0 else 0.0
    stats['pairwise_diff_skewness'] = float(np.nan_to_num(skew(pdist_values), nan=0.0)) if pdist_values.size > 0 else 0.0
    stats['pairwise_diff_kurtosis'] = float(np.nan_to_num(kurtosis(pdist_values), nan=0.0)) if pdist_values.size > 0 else 0.0

    top_diffs = np.sort(pdist_values)[-top_num_diff:] if pdist_values.size > 0 else np.array([])
    for i in range(1, top_num_diff + 1):
        stats[f'top_pairwise_diff_{i}'] = float(top_diffs[-i]) if top_diffs.size >= i else 0.0

    # Denisovan-conditioned human haplotype summaries
    if den_haps is not None:
        den_stats = _compute_denisovan_conditioned_hap_features(
            positions,
            geno,
            den_haps,
            total_length_bp=total_length_bp
        )
        for k, v in den_stats.items():
            stats[k] = v

    return stats


def _compute_diploid_stats(positions, geno, total_length_bp, window_size_bp, top_within_sample, top_num_diff):
    """
    Compute summary statistics for diploid genotype data (values 0, 1, or 2).

    positions: absolute genomic coordinates
    geno:      (n_variants, n_individuals)
    """
    stats = {}
    total_sites, num_indiv = geno.shape
    stats['total_sites'] = total_sites
    stats['num_individuals'] = num_indiv
    stats['num_sites'] = total_sites

    if total_sites == 0:
        stats['mean_allele_frequency'] = 0.0
        stats['var_allele_frequency'] = 0.0
        stats['expected_heterozygosity'] = 0.0
        stats['observed_heterozygosity'] = 0.0
        stats['prop_segregating_sites'] = 0.0
        stats['site_frequency_spectrum'] = [0] * (2 * num_indiv + 1)
        stats['num_singletons'] = 0
        stats['tajimas_d'] = 0.0
        stats['mean_pairwise_diff'] = 0.0
        stats['var_pairwise_diff'] = 0.0
        stats['pairwise_diff_skewness'] = 0.0
        stats['pairwise_diff_kurtosis'] = 0.0
        for i in range(1, top_num_diff + 1):
            stats[f'top_pairwise_diff_{i}'] = 0.0
        stats['mean_within_individual_window_diff'] = 0.0
        stats['var_within_individual_window_diff'] = 0.0
        stats['within_individual_window_diff_skewness'] = 0.0
        stats['within_individual_window_diff_kurtosis'] = 0.0
        for i in range(1, top_within_sample + 1):
            stats[f'top_within_individual_window_diff_{i}'] = 0.0
        return stats

    allele_freq = np.sum(geno, axis=1) / (2 * num_indiv)
    stats['mean_allele_frequency'] = float(np.mean(allele_freq))
    stats['var_allele_frequency'] = float(np.var(allele_freq))

    exp_het = 2 * allele_freq * (1 - allele_freq)
    obs_het = np.mean(geno == 1, axis=1)
    stats['expected_heterozygosity'] = float(np.mean(exp_het))
    stats['observed_heterozygosity'] = float(np.mean(obs_het))

    allele_counts = np.sum(geno, axis=1)
    S_total = np.sum(np.logical_and(allele_counts > 0, allele_counts < 2 * num_indiv))
    stats['prop_segregating_sites'] = float(S_total) / total_sites if total_sites > 0 else 0.0

    sfs = np.bincount(allele_counts.astype(np.int64), minlength=2 * num_indiv + 1)
    stats['site_frequency_spectrum'] = sfs.tolist()
    stats['num_singletons'] = int(sfs[1]) if len(sfs) > 1 else 0

    allele_counts = np.sum(geno, axis=1).astype(int)
    stats['tajimas_d'] = tajimas_d_from_counts(allele_counts, n_chrom=2 * geno.shape[1])

    pdist_values = pdist(geno.T, metric='cityblock')
    stats['mean_pairwise_diff'] = float(pdist_values.mean()) if pdist_values.size > 0 else 0.0
    stats['var_pairwise_diff'] = float(pdist_values.var()) if pdist_values.size > 0 else 0.0
    stats['pairwise_diff_skewness'] = float(np.nan_to_num(skew(pdist_values), nan=0.0)) if pdist_values.size > 0 else 0.0
    stats['pairwise_diff_kurtosis'] = float(np.nan_to_num(kurtosis(pdist_values), nan=0.0)) if pdist_values.size > 0 else 0.0

    top_diffs = np.sort(pdist_values)[-top_num_diff:] if pdist_values.size > 0 else np.array([])
    for i in range(1, top_num_diff + 1):
        stats[f'top_pairwise_diff_{i}'] = float(top_diffs[-i]) if top_diffs.size >= i else 0.0

    # Compare heterozygote fractions across non-empty 10-kb bins in a nominal 50-kb span.
    window_start = np.floor(positions.min() / total_length_bp) * total_length_bp
    window_end = window_start + total_length_bp
    bin_boundaries = np.arange(window_start, window_end + window_size_bp, window_size_bp)

    within_diff = []
    for indiv in range(num_indiv):
        indiv_data = geno[:, indiv]
        window_hets = []
        for i in range(len(bin_boundaries) - 1):
            start = bin_boundaries[i]
            end = bin_boundaries[i + 1]
            idx = np.where((positions >= start) & (positions < end))[0]
            window = indiv_data[idx]
            if window.size > 0:
                window_hets.append(np.mean(window == 1))

        for i in range(len(window_hets)):
            for j in range(i + 1, len(window_hets)):
                within_diff.append(abs(window_hets[i] - window_hets[j]))

    if within_diff:
        within_diff = np.array(within_diff)
        stats['mean_within_individual_window_diff'] = float(within_diff.mean())
        stats['var_within_individual_window_diff'] = float(within_diff.var())
        stats['within_individual_window_diff_skewness'] = float(np.nan_to_num(skew(within_diff), nan=0.0))
        stats['within_individual_window_diff_kurtosis'] = float(np.nan_to_num(kurtosis(within_diff), nan=0.0))
        top_within = np.sort(within_diff)[-top_within_sample:]
        for i in range(1, top_within_sample + 1):
            stats[f'top_within_individual_window_diff_{i}'] = float(top_within[-i]) if top_within.size >= i else 0.0
    else:
        stats['mean_within_individual_window_diff'] = 0.0
        stats['var_within_individual_window_diff'] = 0.0
        stats['within_individual_window_diff_skewness'] = 0.0
        stats['within_individual_window_diff_kurtosis'] = 0.0
        for i in range(1, top_within_sample + 1):
            stats[f'top_within_individual_window_diff_{i}'] = 0.0

    return stats

In [ ]:
import os
import numpy as np
import pandas as pd

def process_chromosome_file(npz_file_path, chromosome):
    """
    Process a single NPZ file for a given chromosome to extract features.

    Parameters:
        npz_file_path (str): Path to the NPZ file.
        chromosome (int): Chromosome number.
    
    Returns:
        pd.DataFrame: DataFrame of features extracted from all arrays in the NPZ file.
                      Includes the window coordinates and chromosome number.
    """
    data = []
    with np.load(npz_file_path, allow_pickle=False) as npz_data:
        # Each archive entry is a matrix for one non-empty 50-kb window.
        for arr_key in npz_data.files:
            # load the matrix for this window
            genotypes = npz_data[arr_key]

            # Exclude positions and the two Denisovan alleles when checking human calls.
            human_only = genotypes[:, 1:-2]
            # if there are no variant alleles (all zeros) in any human column, skip this window
            if not np.any(human_only):
                continue
            stats_both = extract_statistics_both(genotypes)
            if stats_both is None:
                # No polymorphic human sites remain in this window.
                continue
            features = {}
            # Store the NPZ array key.
            features['window'] = arr_key
            
            # Prefix features from the human-only analysis.
            for key, value in stats_both['human_only_stats'].items():
                features[f"human_{key}"] = value
            # Prefix features from the combined (diploid) analysis.
            for key, value in stats_both['combined_stats'].items():
                features[f"combined_{key}"] = value
            
            # Record the chromosome number.
            features['chromosome'] = chromosome
            data.append(features)
        
    return pd.DataFrame(data) if data else pd.DataFrame(columns=["window", "chromosome"])

In [ ]:
# Save one table for each chromosome processed in this run.
FEATURE_DIRECTORY.mkdir(parents=True, exist_ok=True)
feature_files = []
for chrom, matrix_file in sorted(matrix_files.items()):
    features_df = process_chromosome_file(matrix_file, chrom)
    output_csv = FEATURE_DIRECTORY / f"genotype_prediction_features_chr{chrom}.csv"
    features_df.to_csv(output_csv, index=False)
    feature_files.append(output_csv)
    print(f"Chromosome {chrom}: saved {len(features_df)} windows to {output_csv}")

## 3. Merge the chromosome tables

The merged CSV contains one row per retained window, the chromosome number,
window coordinates, and unscaled statistics. Only tables from the current run
are merged. An empty chromosome table contributes no rows.

DEEP selects its 80 model inputs from this table and applies the fitted scaler.
The empirical statistics include `human_num_singletons`; the model inputs
`human_singletons` and `human_high_freq_sites` are separate fields absent here.
The packaged classifier supplies zeros for those two missing inputs.

In [ ]:
def merge_empirical_csv_files(csv_files, output_file):
    """Merge the chromosome tables produced by this run, in chromosome order."""
    data_frames = [pd.read_csv(path) for path in csv_files]
    data_frames = [frame for frame in data_frames if not frame.empty]
    if not data_frames:
        raise ValueError("No windows with human polymorphism were available to merge.")
    merged_df = pd.concat(data_frames, ignore_index=True)
    merged_df.to_csv(output_file, index=False)
    print(f"Saved {len(merged_df)} windows to {output_file}")
    return merged_df


merged_csv = FEATURE_DIRECTORY / "full_empirical_autosomes.csv"
empirical_features = merge_empirical_csv_files(feature_files, merged_csv)
empirical_features.head()

## Use the feature table

The table is ready for `deep predict`. In a terminal, replace the paths with
the merged CSV shown above and your chosen output filename. Select `oceanian`
for the Oceanian panel, or `east-asian` for the Tibetan or Han panel.

```sh
deep predict --population oceanian /path/to/your/full_empirical_autosomes.csv /path/to/your/scores.csv
```